# EasyNCO CVRP Benchmark —— 逐数据 × 逐方法（显式命令、中文注释）

你要做的事情：

- 用 EasyNCO 的命令行：`python EasyNCO/eval.py ...`
- 在 **CVRP** 上评测几种方法
- 数据覆盖三个来源：
  - `EasyNCO/data/datasets/cvrplib-Vrp-Set-XXL`（CVRPLIB XXL，`.vrp/.sol`）
  - `EasyNCO/data/datasets/cross_distribution`（CVRP 1000 的 rotation / explosion，`.pkl`）
  - `EasyNCO/data/datasets/test_dataset_vrp_uniform`（uniform，`.pt`）

评测口径：

- 本 notebook 按各方法 `EasyNCO/settings/<method>_settings.yaml` 的默认流程执行（通常是 `initialization + iteration`）。

输出：

- 每个方法 × 每个数据文件都会产生一份 log：
  - `results/benchmarks/cvrp_datasets/logs/<method>/<dataset_key>.log`

提示：

- 先随便挑一个小规模数据（例如 VRP100）跑通流程，再全量跑。


## 0. 运行前自检

- notebook 建议在仓库根目录运行。
- 每个 `%%bash` cell 会自动用 `git rev-parse --show-toplevel` 切到仓库根目录。

如果你是在没有 `.git` 的环境运行（`git rev-parse` 会失败），请手动把 notebook 的工作目录切到仓库根目录再跑。


In [ ]:
from pathlib import Path

repo = Path.cwd().resolve()
print('当前工作目录:', repo)
print('是否存在 EasyNCO/eval.py:', (repo/'EasyNCO'/'eval.py').exists())
print('EasyNCO datasets root:', repo/'EasyNCO'/'data'/'datasets')
print('预训练 ckpt 目录(应存在):', repo/'EasyNCO'/'pretrained')


## 1. 数据准备：让 EasyNCO 能读到 `cvrplib-Vrp-Set-XXL`

原因：`python EasyNCO/eval.py` 会把 `test_data_path=...` 解析为相对 `EasyNCO/data/datasets/` 的路径。

但 XXL 数据一般放在：`final/data/datasets/cvrplib-Vrp-Set-XXL`（你之前给的数据路径就是这里）。

最省事的做法是：

- 把 `EasyNCO/data/datasets/cvrplib-Vrp-Set-XXL` 做成一个软链接，指向 `final/data/datasets/cvrplib-Vrp-Set-XXL`

下面这个 cell 是幂等的：存在就跳过，不存在才创建。


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

src="final/data/datasets/cvrplib-Vrp-Set-XXL"
dst="EasyNCO/data/datasets/cvrplib-Vrp-Set-XXL"

if [ -e "${dst}" ]; then
  echo "[ok] ${dst} already exists"
elif [ -d "${src}" ]; then
  echo "[link] ${dst} -> ${src}"
  ln -s "${REPO_ROOT}/${src}" "${dst}"
else
  echo "[warn] source not found: ${src}"
  echo "       请把 XXL 数据目录放到 ${dst}，或者修改 src 路径。"
fi

ls -la "EasyNCO/data/datasets" | head -n 50


## 2. CVRP ckpt 准备（你很可能需要做这一步）

这一步是为了避免你 Run All 时直接报“找不到 ckpt”。

- 这里的所有命令默认从 `EasyNCO/pretrained/` 读取模型文件。
- 但本仓库默认几乎只带了 TSP 的 ckpt。

所以：

- 你需要把对应 CVRP 方法的 ckpt 下载/转换后放到 `EasyNCO/pretrained/`。
- 或者你也可以改命令里的：
  - `settings.test_loader.model_dirpath=...`
  - `settings.test_loader.model_filename=...`

下面 cell 会列出当前 `EasyNCO/pretrained/` 里有哪些文件。


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

ls -la EasyNCO/pretrained | sed -n '1,200p'


## 3. 命令参数说明（重点：CVRPLIB 的 capacity_range 过滤）

### (A) 基础参数

- `problem=cvrp`：CVRP 问题。
- `test_data_path=...`：相对 `EasyNCO/data/datasets/` 的路径。
- `settings=<xxx_settings>` + `model=<method>`：选择方法与配置模板。
- `settings.test_loader.model_dirpath=pretrained` + `settings.test_loader.model_filename=...`：指定 ckpt。

### (B) 评测流程（按 settings 默认）


### (C) CVRPLIB 特殊点：默认配置会把 capacity_range 设成 [50, 100]

你之前跑 EasyNCO 时，日志里出现过类似：

- `varying_data_params: {'scale_range': None, ..., 'capacity_range': [50, 100]}`

这对 uniform/cross_distribution 没影响，但对 CVRPLIB loader 会造成“过滤掉所有实例”：

- 例如 XXL 的 CAPACITY=30，会被过滤掉 -> loader 报 `No VRP files found...`。

所以本 notebook 在 CVRPLIB 数据上会显式加：

- `varying_data_params.capacity_range=null`

来关闭过滤。


---
## Dataset: `cvrplib-Vrp-Set-XXL`

- 类型 = `cvrplib`
- scale(命令参数) = **100**
- episodes(仅用于 EasyNCO 统计/进度) = **4**
- 备注：CVRPLIB XXL（超大规模，Antwerp1 DIMENSION=6001）。此类实例对神经方法非常不友好：容易 OOM / 时间不可接受。

CVRPLIB XXL 额外说明：
- 这是一个目录（包含多个 `.vrp`），EasyNCO 会递归读取。
- 如果同名 `.sol` 里有 `Cost ...`，EasyNCO 会把它当作 optimal 并计算 gap。
- 本 notebook 会默认用非常保守的参数：`batch_size=1` + `pomo_size=1` + `aug_factor=1`。


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 说明：OMNI（meta-learning + POMO）。
# 数据：cvrplib-Vrp-Set-XXL
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/omni_cvrp_maml_fomaml.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/omni/cvrplib-Vrp-Set-XXL.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/omni"

ckpt_path="EasyNCO/pretrained/omni_cvrp_maml_fomaml.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/omni/cvrplib-Vrp-Set-XXL.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/omni/cvrplib-Vrp-Set-XXL.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/omni/cvrplib-Vrp-Set-XXL.log"
  exit 0
fi

echo "[run][eval] omni on cvrplib-Vrp-Set-XXL (scale=100, episodes=4, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=cvrp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=cvrplib-Vrp-Set-XXL \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_cvrp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=4 \
  batch_size=1 \
  varying_data_params.capacity_range=null \
  varying_data_params.scale_range=null \
  ++settings.env.pomo_size=1 \
  ++settings.env.aug_factor=1 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/omni/cvrplib-Vrp-Set-XXL.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 说明：LEHD（原生实现含 iteration；本 notebook 按 settings 默认）。
# 数据：cvrplib-Vrp-Set-XXL
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/lehd_cvrp100.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/lehd/cvrplib-Vrp-Set-XXL.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/lehd"

ckpt_path="EasyNCO/pretrained/lehd_cvrp100.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/lehd/cvrplib-Vrp-Set-XXL.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/lehd/cvrplib-Vrp-Set-XXL.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/lehd/cvrplib-Vrp-Set-XXL.log"
  exit 0
fi

echo "[run][eval] lehd on cvrplib-Vrp-Set-XXL (scale=100, episodes=4, batch_size=1, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=cvrp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=cvrplib-Vrp-Set-XXL \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_cvrp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=4 \
  batch_size=1 \
  varying_data_params.capacity_range=null \
  varying_data_params.scale_range=null \
  ++settings.env.pomo_size=1 \
  ++settings.env.aug_factor=1 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/lehd/cvrplib-Vrp-Set-XXL.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 说明：ELG（AR 构造式）。
# 数据：cvrplib-Vrp-Set-XXL
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/elg_cvrp100.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/elg/cvrplib-Vrp-Set-XXL.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/elg"

ckpt_path="EasyNCO/pretrained/elg_cvrp100.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/elg/cvrplib-Vrp-Set-XXL.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/elg/cvrplib-Vrp-Set-XXL.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/elg/cvrplib-Vrp-Set-XXL.log"
  exit 0
fi

echo "[run][eval] elg on cvrplib-Vrp-Set-XXL (scale=100, episodes=4, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=cvrp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=cvrplib-Vrp-Set-XXL \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_cvrp100.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=4 \
  batch_size=1 \
  varying_data_params.capacity_range=null \
  varying_data_params.scale_range=null \
  ++settings.env.pomo_size=1 \
  ++settings.env.aug_factor=1 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/elg/cvrplib-Vrp-Set-XXL.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 说明：ICAM（AR 构造式）。
# 数据：cvrplib-Vrp-Set-XXL
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/icam_cvrp_best.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/icam/cvrplib-Vrp-Set-XXL.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/icam"

ckpt_path="EasyNCO/pretrained/icam_cvrp_best.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/icam/cvrplib-Vrp-Set-XXL.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/icam/cvrplib-Vrp-Set-XXL.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/icam/cvrplib-Vrp-Set-XXL.log"
  exit 0
fi

echo "[run][eval] icam on cvrplib-Vrp-Set-XXL (scale=100, episodes=4, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=cvrp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=cvrplib-Vrp-Set-XXL \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_cvrp_best.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=4 \
  batch_size=1 \
  varying_data_params.capacity_range=null \
  varying_data_params.scale_range=null \
  ++settings.env.pomo_size=1 \
  ++settings.env.aug_factor=1 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/icam/cvrplib-Vrp-Set-XXL.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：dact
# 说明：DACT（这里主要看 greedy 初始化质量）。
# 数据：cvrplib-Vrp-Set-XXL
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/dact_cvrp100.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/dact/cvrplib-Vrp-Set-XXL.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/dact"

ckpt_path="EasyNCO/pretrained/dact_cvrp100.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/dact/cvrplib-Vrp-Set-XXL.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/dact/cvrplib-Vrp-Set-XXL.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/dact/cvrplib-Vrp-Set-XXL.log"
  exit 0
fi

echo "[run][eval] dact on cvrplib-Vrp-Set-XXL (scale=100, episodes=4, batch_size=1, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=dact \
  problem=cvrp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=cvrplib-Vrp-Set-XXL \
  settings=dact_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=dact_cvrp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=4 \
  batch_size=1 \
  varying_data_params.capacity_range=null \
  varying_data_params.scale_range=null \
  ++settings.env.pomo_size=1 \
  ++settings.env.aug_factor=1 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/dact/cvrplib-Vrp-Set-XXL.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：glop
# 说明：GLOP（CVRP 支持；初始化阶段有断言 batch_size=1 / width=1 / n_subset=1）。
# 数据：cvrplib-Vrp-Set-XXL
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/glop_policy_cvrp_1000.pt
# 日志：results/benchmarks/cvrp_datasets/logs/glop/cvrplib-Vrp-Set-XXL.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/glop"

ckpt_path="EasyNCO/pretrained/glop_policy_cvrp_1000.pt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/glop/cvrplib-Vrp-Set-XXL.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/glop/cvrplib-Vrp-Set-XXL.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/glop/cvrplib-Vrp-Set-XXL.log"
  exit 0
fi

echo "[run][eval] glop on cvrplib-Vrp-Set-XXL (scale=100, episodes=4, batch_size=1, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=glop \
  problem=cvrp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=cvrplib-Vrp-Set-XXL \
  settings=glop_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=glop_policy_cvrp_1000.pt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=4 \
  batch_size=1 \
  varying_data_params.capacity_range=null \
  varying_data_params.scale_range=null \
  ++settings.env.pomo_size=1 \
  ++settings.env.aug_factor=1 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/glop/cvrplib-Vrp-Set-XXL.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


---
## Dataset: `cross_distribution/cvrp1000_explosion.pkl`

- 类型 = `cross_distribution`
- scale(命令参数) = **1000**
- episodes(仅用于 EasyNCO 统计/进度) = **2000**
- 备注：cross_distribution：rotation/explosion 分布外泛化（CVRP）。


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 说明：OMNI（meta-learning + POMO）。
# 数据：cross_distribution/cvrp1000_explosion.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/omni_cvrp_maml_fomaml.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/omni/cross_distribution__cvrp1000_explosion.pkl.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/omni"

ckpt_path="EasyNCO/pretrained/omni_cvrp_maml_fomaml.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/omni/cross_distribution__cvrp1000_explosion.pkl.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/omni/cross_distribution__cvrp1000_explosion.pkl.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/omni/cross_distribution__cvrp1000_explosion.pkl.log"
  exit 0
fi

echo "[run][eval] omni on cross_distribution/cvrp1000_explosion.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=cvrp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/cvrp1000_explosion.pkl \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_cvrp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/omni/cross_distribution__cvrp1000_explosion.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 说明：LEHD（原生实现含 iteration；本 notebook 按 settings 默认）。
# 数据：cross_distribution/cvrp1000_explosion.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/lehd_cvrp100.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/lehd/cross_distribution__cvrp1000_explosion.pkl.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/lehd"

ckpt_path="EasyNCO/pretrained/lehd_cvrp100.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/lehd/cross_distribution__cvrp1000_explosion.pkl.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/lehd/cross_distribution__cvrp1000_explosion.pkl.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/lehd/cross_distribution__cvrp1000_explosion.pkl.log"
  exit 0
fi

echo "[run][eval] lehd on cross_distribution/cvrp1000_explosion.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=cvrp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/cvrp1000_explosion.pkl \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_cvrp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/lehd/cross_distribution__cvrp1000_explosion.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 说明：ELG（AR 构造式）。
# 数据：cross_distribution/cvrp1000_explosion.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/elg_cvrp100.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/elg/cross_distribution__cvrp1000_explosion.pkl.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/elg"

ckpt_path="EasyNCO/pretrained/elg_cvrp100.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/elg/cross_distribution__cvrp1000_explosion.pkl.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/elg/cross_distribution__cvrp1000_explosion.pkl.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/elg/cross_distribution__cvrp1000_explosion.pkl.log"
  exit 0
fi

echo "[run][eval] elg on cross_distribution/cvrp1000_explosion.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=cvrp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/cvrp1000_explosion.pkl \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_cvrp100.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/elg/cross_distribution__cvrp1000_explosion.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 说明：ICAM（AR 构造式）。
# 数据：cross_distribution/cvrp1000_explosion.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/icam_cvrp_best.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/icam/cross_distribution__cvrp1000_explosion.pkl.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/icam"

ckpt_path="EasyNCO/pretrained/icam_cvrp_best.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/icam/cross_distribution__cvrp1000_explosion.pkl.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/icam/cross_distribution__cvrp1000_explosion.pkl.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/icam/cross_distribution__cvrp1000_explosion.pkl.log"
  exit 0
fi

echo "[run][eval] icam on cross_distribution/cvrp1000_explosion.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=cvrp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/cvrp1000_explosion.pkl \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_cvrp_best.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/icam/cross_distribution__cvrp1000_explosion.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：dact
# 说明：DACT（这里主要看 greedy 初始化质量）。
# 数据：cross_distribution/cvrp1000_explosion.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/dact_cvrp100.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/dact/cross_distribution__cvrp1000_explosion.pkl.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/dact"

ckpt_path="EasyNCO/pretrained/dact_cvrp100.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/dact/cross_distribution__cvrp1000_explosion.pkl.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/dact/cross_distribution__cvrp1000_explosion.pkl.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/dact/cross_distribution__cvrp1000_explosion.pkl.log"
  exit 0
fi

echo "[run][eval] dact on cross_distribution/cvrp1000_explosion.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=dact \
  problem=cvrp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/cvrp1000_explosion.pkl \
  settings=dact_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=dact_cvrp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/dact/cross_distribution__cvrp1000_explosion.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：glop
# 说明：GLOP（CVRP 支持；初始化阶段有断言 batch_size=1 / width=1 / n_subset=1）。
# 数据：cross_distribution/cvrp1000_explosion.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/glop_policy_cvrp_1000.pt
# 日志：results/benchmarks/cvrp_datasets/logs/glop/cross_distribution__cvrp1000_explosion.pkl.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/glop"

ckpt_path="EasyNCO/pretrained/glop_policy_cvrp_1000.pt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/glop/cross_distribution__cvrp1000_explosion.pkl.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/glop/cross_distribution__cvrp1000_explosion.pkl.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/glop/cross_distribution__cvrp1000_explosion.pkl.log"
  exit 0
fi

echo "[run][eval] glop on cross_distribution/cvrp1000_explosion.pkl (scale=1000, episodes=2000, batch_size=1, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=glop \
  problem=cvrp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/cvrp1000_explosion.pkl \
  settings=glop_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=glop_policy_cvrp_1000.pt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/glop/cross_distribution__cvrp1000_explosion.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


---
## Dataset: `cross_distribution/cvrp1000_rotation.pkl`

- 类型 = `cross_distribution`
- scale(命令参数) = **1000**
- episodes(仅用于 EasyNCO 统计/进度) = **2000**
- 备注：cross_distribution：rotation/explosion 分布外泛化（CVRP）。


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 说明：OMNI（meta-learning + POMO）。
# 数据：cross_distribution/cvrp1000_rotation.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/omni_cvrp_maml_fomaml.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/omni/cross_distribution__cvrp1000_rotation.pkl.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/omni"

ckpt_path="EasyNCO/pretrained/omni_cvrp_maml_fomaml.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/omni/cross_distribution__cvrp1000_rotation.pkl.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/omni/cross_distribution__cvrp1000_rotation.pkl.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/omni/cross_distribution__cvrp1000_rotation.pkl.log"
  exit 0
fi

echo "[run][eval] omni on cross_distribution/cvrp1000_rotation.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=cvrp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/cvrp1000_rotation.pkl \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_cvrp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/omni/cross_distribution__cvrp1000_rotation.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 说明：LEHD（原生实现含 iteration；本 notebook 按 settings 默认）。
# 数据：cross_distribution/cvrp1000_rotation.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/lehd_cvrp100.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/lehd/cross_distribution__cvrp1000_rotation.pkl.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/lehd"

ckpt_path="EasyNCO/pretrained/lehd_cvrp100.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/lehd/cross_distribution__cvrp1000_rotation.pkl.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/lehd/cross_distribution__cvrp1000_rotation.pkl.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/lehd/cross_distribution__cvrp1000_rotation.pkl.log"
  exit 0
fi

echo "[run][eval] lehd on cross_distribution/cvrp1000_rotation.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=cvrp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/cvrp1000_rotation.pkl \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_cvrp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/lehd/cross_distribution__cvrp1000_rotation.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 说明：ELG（AR 构造式）。
# 数据：cross_distribution/cvrp1000_rotation.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/elg_cvrp100.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/elg/cross_distribution__cvrp1000_rotation.pkl.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/elg"

ckpt_path="EasyNCO/pretrained/elg_cvrp100.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/elg/cross_distribution__cvrp1000_rotation.pkl.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/elg/cross_distribution__cvrp1000_rotation.pkl.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/elg/cross_distribution__cvrp1000_rotation.pkl.log"
  exit 0
fi

echo "[run][eval] elg on cross_distribution/cvrp1000_rotation.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=cvrp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/cvrp1000_rotation.pkl \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_cvrp100.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/elg/cross_distribution__cvrp1000_rotation.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 说明：ICAM（AR 构造式）。
# 数据：cross_distribution/cvrp1000_rotation.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/icam_cvrp_best.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/icam/cross_distribution__cvrp1000_rotation.pkl.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/icam"

ckpt_path="EasyNCO/pretrained/icam_cvrp_best.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/icam/cross_distribution__cvrp1000_rotation.pkl.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/icam/cross_distribution__cvrp1000_rotation.pkl.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/icam/cross_distribution__cvrp1000_rotation.pkl.log"
  exit 0
fi

echo "[run][eval] icam on cross_distribution/cvrp1000_rotation.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=cvrp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/cvrp1000_rotation.pkl \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_cvrp_best.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/icam/cross_distribution__cvrp1000_rotation.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：dact
# 说明：DACT（这里主要看 greedy 初始化质量）。
# 数据：cross_distribution/cvrp1000_rotation.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/dact_cvrp100.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/dact/cross_distribution__cvrp1000_rotation.pkl.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/dact"

ckpt_path="EasyNCO/pretrained/dact_cvrp100.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/dact/cross_distribution__cvrp1000_rotation.pkl.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/dact/cross_distribution__cvrp1000_rotation.pkl.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/dact/cross_distribution__cvrp1000_rotation.pkl.log"
  exit 0
fi

echo "[run][eval] dact on cross_distribution/cvrp1000_rotation.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=dact \
  problem=cvrp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/cvrp1000_rotation.pkl \
  settings=dact_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=dact_cvrp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/dact/cross_distribution__cvrp1000_rotation.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：glop
# 说明：GLOP（CVRP 支持；初始化阶段有断言 batch_size=1 / width=1 / n_subset=1）。
# 数据：cross_distribution/cvrp1000_rotation.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/glop_policy_cvrp_1000.pt
# 日志：results/benchmarks/cvrp_datasets/logs/glop/cross_distribution__cvrp1000_rotation.pkl.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/glop"

ckpt_path="EasyNCO/pretrained/glop_policy_cvrp_1000.pt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/glop/cross_distribution__cvrp1000_rotation.pkl.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/glop/cross_distribution__cvrp1000_rotation.pkl.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/glop/cross_distribution__cvrp1000_rotation.pkl.log"
  exit 0
fi

echo "[run][eval] glop on cross_distribution/cvrp1000_rotation.pkl (scale=1000, episodes=2000, batch_size=1, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=glop \
  problem=cvrp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/cvrp1000_rotation.pkl \
  settings=glop_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=glop_policy_cvrp_1000.pt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/glop/cross_distribution__cvrp1000_rotation.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


---
## Dataset: `test_dataset_vrp_uniform/test_vrp100_capacity50_nums10000_uniform.pt`

- 类型 = `uniform`
- scale(命令参数) = **100**
- episodes(仅用于 EasyNCO 统计/进度) = **10000**
- 备注：uniform：同分布随机 CVRP（文件名含 capacity / nums）。

- 文件名中的 capacity=50 仅用于描述；demand 是否已归一化取决于数据生成。


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 说明：OMNI（meta-learning + POMO）。
# 数据：test_dataset_vrp_uniform/test_vrp100_capacity50_nums10000_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/omni_cvrp_maml_fomaml.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/omni/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/omni"

ckpt_path="EasyNCO/pretrained/omni_cvrp_maml_fomaml.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/omni/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/omni/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/omni/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"
  exit 0
fi

echo "[run][eval] omni on test_dataset_vrp_uniform/test_vrp100_capacity50_nums10000_uniform.pt (scale=100, episodes=10000, batch_size=128, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=cvrp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp100_capacity50_nums10000_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_cvrp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=10000 \
  batch_size=128 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/omni/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 说明：LEHD（原生实现含 iteration；本 notebook 按 settings 默认）。
# 数据：test_dataset_vrp_uniform/test_vrp100_capacity50_nums10000_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/lehd_cvrp100.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/lehd/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/lehd"

ckpt_path="EasyNCO/pretrained/lehd_cvrp100.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/lehd/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/lehd/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/lehd/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"
  exit 0
fi

echo "[run][eval] lehd on test_dataset_vrp_uniform/test_vrp100_capacity50_nums10000_uniform.pt (scale=100, episodes=10000, batch_size=128, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=cvrp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp100_capacity50_nums10000_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_cvrp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=10000 \
  batch_size=128 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/lehd/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 说明：ELG（AR 构造式）。
# 数据：test_dataset_vrp_uniform/test_vrp100_capacity50_nums10000_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/elg_cvrp100.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/elg/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/elg"

ckpt_path="EasyNCO/pretrained/elg_cvrp100.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/elg/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/elg/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/elg/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"
  exit 0
fi

echo "[run][eval] elg on test_dataset_vrp_uniform/test_vrp100_capacity50_nums10000_uniform.pt (scale=100, episodes=10000, batch_size=128, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=cvrp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp100_capacity50_nums10000_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_cvrp100.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=10000 \
  batch_size=128 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/elg/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 说明：ICAM（AR 构造式）。
# 数据：test_dataset_vrp_uniform/test_vrp100_capacity50_nums10000_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/icam_cvrp_best.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/icam/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/icam"

ckpt_path="EasyNCO/pretrained/icam_cvrp_best.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/icam/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/icam/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/icam/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"
  exit 0
fi

echo "[run][eval] icam on test_dataset_vrp_uniform/test_vrp100_capacity50_nums10000_uniform.pt (scale=100, episodes=10000, batch_size=128, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=cvrp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp100_capacity50_nums10000_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_cvrp_best.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=10000 \
  batch_size=128 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/icam/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：dact
# 说明：DACT（这里主要看 greedy 初始化质量）。
# 数据：test_dataset_vrp_uniform/test_vrp100_capacity50_nums10000_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/dact_cvrp100.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/dact/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/dact"

ckpt_path="EasyNCO/pretrained/dact_cvrp100.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/dact/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/dact/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/dact/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"
  exit 0
fi

echo "[run][eval] dact on test_dataset_vrp_uniform/test_vrp100_capacity50_nums10000_uniform.pt (scale=100, episodes=10000, batch_size=128, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=dact \
  problem=cvrp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp100_capacity50_nums10000_uniform.pt \
  settings=dact_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=dact_cvrp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=10000 \
  batch_size=128 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/dact/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：glop
# 说明：GLOP（CVRP 支持；初始化阶段有断言 batch_size=1 / width=1 / n_subset=1）。
# 数据：test_dataset_vrp_uniform/test_vrp100_capacity50_nums10000_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/glop_policy_cvrp_1000.pt
# 日志：results/benchmarks/cvrp_datasets/logs/glop/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/glop"

ckpt_path="EasyNCO/pretrained/glop_policy_cvrp_1000.pt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/glop/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/glop/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/glop/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"
  exit 0
fi

echo "[run][eval] glop on test_dataset_vrp_uniform/test_vrp100_capacity50_nums10000_uniform.pt (scale=100, episodes=10000, batch_size=1, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=glop \
  problem=cvrp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp100_capacity50_nums10000_uniform.pt \
  settings=glop_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=glop_policy_cvrp_1000.pt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=10000 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/glop/test_dataset_vrp_uniform__test_vrp100_capacity50_nums10000_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


---
## Dataset: `test_dataset_vrp_uniform/test_vrp200_capacity80_nums128_uniform.pt`

- 类型 = `uniform`
- scale(命令参数) = **200**
- episodes(仅用于 EasyNCO 统计/进度) = **128**
- 备注：uniform：同分布随机 CVRP（文件名含 capacity / nums）。

- 文件名中的 capacity=80 仅用于描述；demand 是否已归一化取决于数据生成。


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 说明：OMNI（meta-learning + POMO）。
# 数据：test_dataset_vrp_uniform/test_vrp200_capacity80_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/omni_cvrp_maml_fomaml.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/omni/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/omni"

ckpt_path="EasyNCO/pretrained/omni_cvrp_maml_fomaml.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/omni/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/omni/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/omni/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"
  exit 0
fi

echo "[run][eval] omni on test_dataset_vrp_uniform/test_vrp200_capacity80_nums128_uniform.pt (scale=200, episodes=128, batch_size=64, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=cvrp \
  scale=200 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp200_capacity80_nums128_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_cvrp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=64 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/omni/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 说明：LEHD（原生实现含 iteration；本 notebook 按 settings 默认）。
# 数据：test_dataset_vrp_uniform/test_vrp200_capacity80_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/lehd_cvrp100.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/lehd/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/lehd"

ckpt_path="EasyNCO/pretrained/lehd_cvrp100.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/lehd/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/lehd/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/lehd/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"
  exit 0
fi

echo "[run][eval] lehd on test_dataset_vrp_uniform/test_vrp200_capacity80_nums128_uniform.pt (scale=200, episodes=128, batch_size=64, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=cvrp \
  scale=200 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp200_capacity80_nums128_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_cvrp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=64 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/lehd/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 说明：ELG（AR 构造式）。
# 数据：test_dataset_vrp_uniform/test_vrp200_capacity80_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/elg_cvrp100.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/elg/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/elg"

ckpt_path="EasyNCO/pretrained/elg_cvrp100.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/elg/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/elg/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/elg/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"
  exit 0
fi

echo "[run][eval] elg on test_dataset_vrp_uniform/test_vrp200_capacity80_nums128_uniform.pt (scale=200, episodes=128, batch_size=64, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=cvrp \
  scale=200 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp200_capacity80_nums128_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_cvrp100.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=64 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/elg/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 说明：ICAM（AR 构造式）。
# 数据：test_dataset_vrp_uniform/test_vrp200_capacity80_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/icam_cvrp_best.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/icam/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/icam"

ckpt_path="EasyNCO/pretrained/icam_cvrp_best.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/icam/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/icam/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/icam/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"
  exit 0
fi

echo "[run][eval] icam on test_dataset_vrp_uniform/test_vrp200_capacity80_nums128_uniform.pt (scale=200, episodes=128, batch_size=64, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=cvrp \
  scale=200 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp200_capacity80_nums128_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_cvrp_best.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=64 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/icam/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：dact
# 说明：DACT（这里主要看 greedy 初始化质量）。
# 数据：test_dataset_vrp_uniform/test_vrp200_capacity80_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/dact_cvrp100.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/dact/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/dact"

ckpt_path="EasyNCO/pretrained/dact_cvrp100.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/dact/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/dact/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/dact/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"
  exit 0
fi

echo "[run][eval] dact on test_dataset_vrp_uniform/test_vrp200_capacity80_nums128_uniform.pt (scale=200, episodes=128, batch_size=64, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=dact \
  problem=cvrp \
  scale=200 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp200_capacity80_nums128_uniform.pt \
  settings=dact_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=dact_cvrp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=64 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/dact/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：glop
# 说明：GLOP（CVRP 支持；初始化阶段有断言 batch_size=1 / width=1 / n_subset=1）。
# 数据：test_dataset_vrp_uniform/test_vrp200_capacity80_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/glop_policy_cvrp_1000.pt
# 日志：results/benchmarks/cvrp_datasets/logs/glop/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/glop"

ckpt_path="EasyNCO/pretrained/glop_policy_cvrp_1000.pt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/glop/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/glop/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/glop/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"
  exit 0
fi

echo "[run][eval] glop on test_dataset_vrp_uniform/test_vrp200_capacity80_nums128_uniform.pt (scale=200, episodes=128, batch_size=1, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=glop \
  problem=cvrp \
  scale=200 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp200_capacity80_nums128_uniform.pt \
  settings=glop_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=glop_policy_cvrp_1000.pt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/glop/test_dataset_vrp_uniform__test_vrp200_capacity80_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


---
## Dataset: `test_dataset_vrp_uniform/test_vrp500_capacity100_nums128_uniform.pt`

- 类型 = `uniform`
- scale(命令参数) = **500**
- episodes(仅用于 EasyNCO 统计/进度) = **128**
- 备注：uniform：同分布随机 CVRP（文件名含 capacity / nums）。

- 文件名中的 capacity=100 仅用于描述；demand 是否已归一化取决于数据生成。


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 说明：OMNI（meta-learning + POMO）。
# 数据：test_dataset_vrp_uniform/test_vrp500_capacity100_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/omni_cvrp_maml_fomaml.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/omni/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/omni"

ckpt_path="EasyNCO/pretrained/omni_cvrp_maml_fomaml.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/omni/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/omni/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/omni/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"
  exit 0
fi

echo "[run][eval] omni on test_dataset_vrp_uniform/test_vrp500_capacity100_nums128_uniform.pt (scale=500, episodes=128, batch_size=16, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=cvrp \
  scale=500 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp500_capacity100_nums128_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_cvrp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/omni/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 说明：LEHD（原生实现含 iteration；本 notebook 按 settings 默认）。
# 数据：test_dataset_vrp_uniform/test_vrp500_capacity100_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/lehd_cvrp100.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/lehd/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/lehd"

ckpt_path="EasyNCO/pretrained/lehd_cvrp100.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/lehd/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/lehd/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/lehd/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"
  exit 0
fi

echo "[run][eval] lehd on test_dataset_vrp_uniform/test_vrp500_capacity100_nums128_uniform.pt (scale=500, episodes=128, batch_size=16, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=cvrp \
  scale=500 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp500_capacity100_nums128_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_cvrp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/lehd/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 说明：ELG（AR 构造式）。
# 数据：test_dataset_vrp_uniform/test_vrp500_capacity100_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/elg_cvrp100.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/elg/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/elg"

ckpt_path="EasyNCO/pretrained/elg_cvrp100.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/elg/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/elg/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/elg/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"
  exit 0
fi

echo "[run][eval] elg on test_dataset_vrp_uniform/test_vrp500_capacity100_nums128_uniform.pt (scale=500, episodes=128, batch_size=16, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=cvrp \
  scale=500 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp500_capacity100_nums128_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_cvrp100.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/elg/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 说明：ICAM（AR 构造式）。
# 数据：test_dataset_vrp_uniform/test_vrp500_capacity100_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/icam_cvrp_best.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/icam/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/icam"

ckpt_path="EasyNCO/pretrained/icam_cvrp_best.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/icam/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/icam/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/icam/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"
  exit 0
fi

echo "[run][eval] icam on test_dataset_vrp_uniform/test_vrp500_capacity100_nums128_uniform.pt (scale=500, episodes=128, batch_size=16, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=cvrp \
  scale=500 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp500_capacity100_nums128_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_cvrp_best.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/icam/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：dact
# 说明：DACT（这里主要看 greedy 初始化质量）。
# 数据：test_dataset_vrp_uniform/test_vrp500_capacity100_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/dact_cvrp100.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/dact/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/dact"

ckpt_path="EasyNCO/pretrained/dact_cvrp100.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/dact/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/dact/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/dact/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"
  exit 0
fi

echo "[run][eval] dact on test_dataset_vrp_uniform/test_vrp500_capacity100_nums128_uniform.pt (scale=500, episodes=128, batch_size=16, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=dact \
  problem=cvrp \
  scale=500 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp500_capacity100_nums128_uniform.pt \
  settings=dact_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=dact_cvrp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/dact/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：glop
# 说明：GLOP（CVRP 支持；初始化阶段有断言 batch_size=1 / width=1 / n_subset=1）。
# 数据：test_dataset_vrp_uniform/test_vrp500_capacity100_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/glop_policy_cvrp_1000.pt
# 日志：results/benchmarks/cvrp_datasets/logs/glop/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/glop"

ckpt_path="EasyNCO/pretrained/glop_policy_cvrp_1000.pt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/glop/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/glop/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/glop/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"
  exit 0
fi

echo "[run][eval] glop on test_dataset_vrp_uniform/test_vrp500_capacity100_nums128_uniform.pt (scale=500, episodes=128, batch_size=1, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=glop \
  problem=cvrp \
  scale=500 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp500_capacity100_nums128_uniform.pt \
  settings=glop_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=glop_policy_cvrp_1000.pt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/glop/test_dataset_vrp_uniform__test_vrp500_capacity100_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


---
## Dataset: `test_dataset_vrp_uniform/test_vrp1000_capacity250_nums128_uniform.pt`

- 类型 = `uniform`
- scale(命令参数) = **1000**
- episodes(仅用于 EasyNCO 统计/进度) = **128**
- 备注：uniform：同分布随机 CVRP（文件名含 capacity / nums）。

- 文件名中的 capacity=250 仅用于描述；demand 是否已归一化取决于数据生成。


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 说明：OMNI（meta-learning + POMO）。
# 数据：test_dataset_vrp_uniform/test_vrp1000_capacity250_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/omni_cvrp_maml_fomaml.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/omni/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/omni"

ckpt_path="EasyNCO/pretrained/omni_cvrp_maml_fomaml.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/omni/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/omni/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/omni/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"
  exit 0
fi

echo "[run][eval] omni on test_dataset_vrp_uniform/test_vrp1000_capacity250_nums128_uniform.pt (scale=1000, episodes=128, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=cvrp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp1000_capacity250_nums128_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_cvrp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/omni/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 说明：LEHD（原生实现含 iteration；本 notebook 按 settings 默认）。
# 数据：test_dataset_vrp_uniform/test_vrp1000_capacity250_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/lehd_cvrp100.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/lehd/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/lehd"

ckpt_path="EasyNCO/pretrained/lehd_cvrp100.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/lehd/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/lehd/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/lehd/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"
  exit 0
fi

echo "[run][eval] lehd on test_dataset_vrp_uniform/test_vrp1000_capacity250_nums128_uniform.pt (scale=1000, episodes=128, batch_size=4, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=cvrp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp1000_capacity250_nums128_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_cvrp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/lehd/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 说明：ELG（AR 构造式）。
# 数据：test_dataset_vrp_uniform/test_vrp1000_capacity250_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/elg_cvrp100.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/elg/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/elg"

ckpt_path="EasyNCO/pretrained/elg_cvrp100.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/elg/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/elg/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/elg/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"
  exit 0
fi

echo "[run][eval] elg on test_dataset_vrp_uniform/test_vrp1000_capacity250_nums128_uniform.pt (scale=1000, episodes=128, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=cvrp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp1000_capacity250_nums128_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_cvrp100.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/elg/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 说明：ICAM（AR 构造式）。
# 数据：test_dataset_vrp_uniform/test_vrp1000_capacity250_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/icam_cvrp_best.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/icam/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/icam"

ckpt_path="EasyNCO/pretrained/icam_cvrp_best.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/icam/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/icam/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/icam/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"
  exit 0
fi

echo "[run][eval] icam on test_dataset_vrp_uniform/test_vrp1000_capacity250_nums128_uniform.pt (scale=1000, episodes=128, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=cvrp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp1000_capacity250_nums128_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_cvrp_best.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/icam/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：dact
# 说明：DACT（这里主要看 greedy 初始化质量）。
# 数据：test_dataset_vrp_uniform/test_vrp1000_capacity250_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/dact_cvrp100.ckpt
# 日志：results/benchmarks/cvrp_datasets/logs/dact/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/dact"

ckpt_path="EasyNCO/pretrained/dact_cvrp100.ckpt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/dact/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/dact/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/dact/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"
  exit 0
fi

echo "[run][eval] dact on test_dataset_vrp_uniform/test_vrp1000_capacity250_nums128_uniform.pt (scale=1000, episodes=128, batch_size=4, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=dact \
  problem=cvrp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp1000_capacity250_nums128_uniform.pt \
  settings=dact_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=dact_cvrp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/dact/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：glop
# 说明：GLOP（CVRP 支持；初始化阶段有断言 batch_size=1 / width=1 / n_subset=1）。
# 数据：test_dataset_vrp_uniform/test_vrp1000_capacity250_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/glop_policy_cvrp_1000.pt
# 日志：results/benchmarks/cvrp_datasets/logs/glop/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log

mkdir -p "results/benchmarks/cvrp_datasets/logs/glop"

ckpt_path="EasyNCO/pretrained/glop_policy_cvrp_1000.pt"
if [ ! -f "${ckpt_path}" ]; then
  echo "[ERROR] missing ckpt: ${ckpt_path}" | tee "results/benchmarks/cvrp_datasets/logs/glop/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"
  echo "你需要把对应 CVRP ckpt 放到 EasyNCO/pretrained/，或者修改本 cell 里的 model_filename。" | tee -a "results/benchmarks/cvrp_datasets/logs/glop/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"
  echo "（为保证 notebook 可以继续往下跑，这里 exit 0）" | tee -a "results/benchmarks/cvrp_datasets/logs/glop/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"
  exit 0
fi

echo "[run][eval] glop on test_dataset_vrp_uniform/test_vrp1000_capacity250_nums128_uniform.pt (scale=1000, episodes=128, batch_size=1, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=glop \
  problem=cvrp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_vrp_uniform/test_vrp1000_capacity250_nums128_uniform.pt \
  settings=glop_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=glop_policy_cvrp_1000.pt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/cvrp_datasets/logs/glop/test_dataset_vrp_uniform__test_vrp1000_capacity250_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0
